# Python Intermediate — AI Engineering Interview Prep

Covers: OOP, dataclasses, decorators, generators, context managers, type hints.

## 1. Object-Oriented Programming

In [ ]:
class Dog:
    species = "Canis familiaris"   # class variable

    def __init__(self, name: str, age: int):
        self.name = name
        self._age = age            # convention: single _ = protected

    def __repr__(self):
        return f"Dog(name={self.name!r}, age={self._age})"

    def __str__(self):
        return f"{self.name} ({self._age} yrs)"

    def __eq__(self, other):
        return isinstance(other, Dog) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if value < 0:
            raise ValueError("Age cannot be negative")
        self._age = value

d = Dog("Rex", 3)
print(repr(d))
print(str(d))
d.age = 4
print(d.age)
print(Dog("Rex", 5) == Dog("Rex", 3))  # True — equality by name

In [ ]:
# Inheritance, super(), MRO
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        raise NotImplementedError

class Cat(Animal):
    def speak(self):
        return "Meow"

class GuideDog(Dog, Animal):
    def __init__(self, name, age, owner):
        super().__init__(name, age)
        self.owner = owner
    def speak(self):
        return "Woof (guide)"

print(GuideDog.__mro__)   # Method Resolution Order
gd = GuideDog("Buddy", 2, "Alice")
print(gd.speak())

In [ ]:
# @classmethod (alternative constructor) and @staticmethod
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, k):
        return cls(k - 273.15)

    @staticmethod
    def is_freezing(celsius):
        return celsius <= 0

    def __repr__(self):
        return f"{self.celsius:.1f}°C"

t = Temperature.from_fahrenheit(212)
print(t)
print(Temperature.is_freezing(-5))

## 2. Dataclasses

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Employee:
    name: str
    department: str
    salary: float
    skills: List[str] = field(default_factory=list)
    _id: int = field(default=0, repr=False)

    def __post_init__(self):
        if self.salary < 0:
            raise ValueError("Salary must be non-negative")
        self.name = self.name.title()

    def add_skill(self, skill: str):
        self.skills.append(skill)

e1 = Employee("alice", "Engineering", 95000)
e1.add_skill("Python")
e2 = Employee("alice", "Engineering", 95000)
print(e1)
print(e1 == e2)   # auto-generated __eq__

@dataclass(frozen=True, order=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(sorted([Point(3,1), Point(1,2), Point(2,0)]))

## 3. Decorators

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)     # preserves __name__, __doc__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

def memoize(func):
    cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]
    return wrapper

@timer
@memoize
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(fib(30))
print(fib(30))  # cached — much faster second call

In [ ]:
# Retry decorator with exponential backoff
import random

def retry(max_attempts=3, delay=0.1, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts:
                        raise
                    wait = delay * (2 ** (attempt - 1))
                    print(f"Attempt {attempt} failed: {e}. Retrying in {wait:.2f}s")
                    time.sleep(wait)
        return wrapper
    return decorator

@retry(max_attempts=4, delay=0.01)
def flaky_api():
    if random.random() < 0.7:
        raise ConnectionError("Simulated network failure")
    return "success"

try:
    print(flaky_api())
except ConnectionError:
    print("All attempts failed")

## 4. Generators

In [ ]:
# Generator function
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

import itertools
fib_gen = fibonacci()
first_10 = list(itertools.islice(fib_gen, 10))
print("First 10 Fibonacci:", first_10)

# Generator expression vs list — memory comparison
import sys
gen_expr = (x**2 for x in range(10000))
list_comp = [x**2 for x in range(10000)]
print(f"Generator size: {sys.getsizeof(gen_expr)} bytes")
print(f"List size:      {sys.getsizeof(list_comp)} bytes")

In [ ]:
# itertools — powerful combinatorial tools
from itertools import chain, groupby, product, combinations, permutations

# chain: flatten iterables
merged = list(chain([1,2], [3,4], [5]))
print("chain:", merged)

# groupby: consecutive groups (data must be sorted!)
data = sorted(["apple", "avocado", "banana", "blueberry", "cherry"], key=lambda x: x[0])
for letter, group in groupby(data, key=lambda x: x[0]):
    print(f"{letter}: {list(group)}")

# combinations and permutations
print("C(4,2):", list(combinations(range(4), 2)))
print("P(3,2):", list(permutations(range(3), 2)))

## 5. Context Managers

In [ ]:
# Class-based context manager
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Elapsed: {self.elapsed:.4f}s")
        return False  # don't suppress exceptions

with Timer() as t:
    total = sum(range(1_000_000))
print(f"Sum: {total}")

In [ ]:
# @contextmanager — simpler approach
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    try:
        yield {"resource": name, "status": "open"}
    except Exception as e:
        print(f"Error during use of {name}: {e}")
        raise
    finally:
        print(f"Releasing {name}")

with managed_resource("DB connection") as res:
    print(f"Using: {res}")

## 6. Type Hints

In [ ]:
from typing import Optional, Union, List, Dict, Tuple, Callable, TypeVar, Generic

T = TypeVar("T")

class Stack(Generic[T]):
    def __init__(self):
        self._items: List[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> Optional[T]:
        return self._items.pop() if self._items else None

    def peek(self) -> Optional[T]:
        return self._items[-1] if self._items else None

    def __len__(self) -> int:
        return len(self._items)

stack: Stack[int] = Stack()
for x in [1, 2, 3]:
    stack.push(x)
print(f"peek={stack.peek()}, pop={stack.pop()}, len={len(stack)}")

In [ ]:
# Practical type hint patterns
def process(
    data: List[Dict[str, Union[int, float, str]]],
    transform: Optional[Callable[[float], float]] = None
) -> Tuple[float, float]:
    values = [float(row.get("value", 0)) for row in data]
    if transform:
        values = [transform(v) for v in values]
    return min(values), max(values)

data = [{"value": 10}, {"value": 25}, {"value": 5}]
print(process(data))
print(process(data, transform=lambda x: x**0.5))

## Interview Tips

- Use `@dataclass(frozen=True)` for immutable value objects (hashable, can use in sets/dicts).
- `functools.wraps` is essential in decorators — without it, `func.__name__` returns `'wrapper'`.
- Generators are lazy: use when data is large or infinite. `itertools.islice` for bounded take.
- Context managers guarantee cleanup even on exception — critical for resources (files, connections).
- `TypeVar` + `Generic` enables reusable typed containers (like Stack[T]).
- MRO (Method Resolution Order) follows C3 linearization — know `ClassName.__mro__` for debugging.